# Proyecto: Mercado de Limones Argentinos — Inferencia Causal
## Notebook 02 — Ingesta y validación serie temporal: Inscripciones Iniciales (0km)
### Período: Enero 2023 — Mayo 2026 (41 meses)
### Fecha: Junio 2026
### Autor: Rodolfo Gabriel Riveros Lobos

---

## Objetivo

Ejecutar el pipeline ETL sobre los 41 archivos CSV de inscripciones iniciales
DNRPA y construir la serie temporal mensual agregada (`serie_0km.parquet`).

**Output:** `data/processed/serie_0km.parquet` — 41 filas, 7 columnas.

---

## Rol de esta serie en el proyecto

La serie de 0km es la **variable predictora** del modelo causal. La hipótesis
central del proyecto es que el volumen de inscripciones iniciales (proxy del
sobrestock en concesionarias) antecede y explica el deterioro del ISA en el
mercado de usados (Test de Granger, Notebook 03).

Esta serie actúa como el **indicador adelantado**: el 0km hoy → el limón de usados en t+k meses.

---

## Diferencias metodológicas respecto al Notebook 01

| Dimensión | Usados (NB01) | 0km (NB02) |
|---|---|---|
| Dataset DNRPA | `dnrpa-transferencias-autos-` | `dnrpa-inscripciones-iniciales-autos-` |
| Directorio | `Raw_Usados/` | `Raw_0km/` |
| `tipo` en pipeline | `'usados'` | `'0km'` |
| Trámites excluidos | 1 filtro (`SUBASTADOS`) | 3 filtros (`FORM. 05`, `CLASICO`, `DICTAMEN`) |
| Output | `serie_usados.parquet` | `serie_0km.parquet` |
| Interpretación de `antiguedad_media` | Proxy de selección adversa | No aplica como ISA: las inscripciones son todas modelo del año |
| Métrica clave | `pct_protocolo21`, `antiguedad_media` | **`total`** (volumen mensual de 0km = proxy de sobrestock) |

---

## Limitaciones activas en esta fase

- **Ruido transaccional:** Los 3 filtros de `TRAMITES_EXCLUIDOS_0KM` son críticos.
  El `FORM. 05` (re-empadronamientos) y `CLASICO` / `DICTAMEN` no responden
  a demanda comercial real y contaminarían el volumen como proxy de sobrestock.
- **Subregistro en concesionarias:** Una fracción de operaciones entre
  agencieros puede registrarse bajo categorías simplificadas.
  Este notebook no puede corregirlo; se documenta como limitación del modelo.
- **`antiguedad_media` en 0km:** Todas las inscripciones iniciales deberían
  tener antigüedad ≈ 0. Si se observan valores altos, indica registros
  tardíos o errores en `automotor_anio_modelo` — se verificará en la validación.
- **Punto ciego de Akerlof:** El dataset registra flujo de salida de fábrica/
  importadora, no intención de compra no realizada. El volumen es proxy,
  no medición directa de sobrestock.

---
## Setup

In [1]:
import sys
from pathlib import Path

# Agrega src/ al path para poder importar pipeline.py
sys.path.insert(0, str(Path.cwd().parent / "src"))

from pipeline import construir_serie
from config import PATHS, TRAMITES_EXCLUIDOS_0KM

print("Imports OK")
print(f"Raw_0km:  {PATHS['raw_0km']}")
print(f"Existe:   {PATHS['raw_0km'].exists()}")
print()
print("Filtros activos para 0km:")
for f in TRAMITES_EXCLUIDOS_0KM:
    print(f"  - '{f}'")

Imports OK
Raw_0km:  c:\Users\usuario\Documents\PORTFOLIO\mercado-limones-causal\data\Raw_0km
Existe:   True

Filtros activos para 0km:
  - 'FORM. 05'
  - 'CLASICO'
  - 'DICTAMEN'


---
## Construcción de la serie temporal

In [2]:
serie_0km = construir_serie("0km")

CONSTRUYENDO SERIE — 0KM
Archivos encontrados: 41
2023-01 →     50,617 originales →   50,601 limpios (100.0%)
2023-02 →     30,704 originales →   30,690 limpios (100.0%)
2023-03 →     40,128 originales →   40,095 limpios (99.9%)
2023-04 →     34,963 originales →   34,929 limpios (99.9%)
2023-05 →     40,361 originales →   40,317 limpios (99.9%)
2023-06 →     40,302 originales →   40,267 limpios (99.9%)
2023-07 →     44,326 originales →   44,283 limpios (99.9%)
2023-08 →     39,702 originales →   39,655 limpios (99.9%)
2023-09 →     33,843 originales →   33,785 limpios (99.8%)
2023-10 →     42,151 originales →   42,094 limpios (99.9%)
2023-11 →     36,214 originales →   36,156 limpios (99.8%)
2023-12 →     18,796 originales →   18,760 limpios (99.8%)
2024-01 →     34,166 originales →   34,146 limpios (99.9%)
2024-02 →     25,213 originales →   25,195 limpios (99.9%)
2024-03 →     25,975 originales →   25,949 limpios (99.9%)
2024-04 →     33,123 originales →   33,094 limpios (99.9%)
2024

---
## Inspección de la serie

In [3]:
print(f"Shape: {serie_0km.shape}")
print(f"\nPrimer período:  {serie_0km['periodo'].iloc[0]}")
print(f"Último período:  {serie_0km['periodo'].iloc[-1]}")
print(f"\nSerie completa:")
print(serie_0km.to_string())

Shape: (41, 7)

Primer período:  2023-01
Último período:  2026-05

Serie completa:
    periodo  anio  mes  total  antiguedad_media  pct_protocolo21  pct_jovenes
0   2023-01  2023    1  50601            0.1272          28.9658      99.6225
1   2023-02  2023    2  30690            0.1156          27.0055      99.5275
2   2023-03  2023    3  40095            0.1102          29.1533      99.5361
3   2023-04  2023    4  34929            0.0840          27.7649      99.6564
4   2023-05  2023    5  40317            0.0721          27.0134      99.6652
5   2023-06  2023    6  40267            0.0674          29.8905      99.6598
6   2023-07  2023    7  44283            0.0695          28.0762      99.6477
7   2023-08  2023    8  39655            0.0696          24.3097      99.5461
8   2023-09  2023    9  33785            0.0497          17.2917      99.5767
9   2023-10  2023   10  42094            0.0501          28.9804      99.6365
10  2023-11  2023   11  36156            0.0535          27

---
## Validación — Diagnóstico de la serie

In [9]:
import pandas as pd

print("=" * 60)
print("VALIDACIÓN — SERIE 0KM")
print("=" * 60)

# 1. Continuidad mensual
periodos = pd.to_datetime(serie_0km['periodo'], format='%Y-%m')
gaps = periodos.diff().dropna()
meses_con_gap = gaps[gaps > pd.Timedelta(days=32)]

if len(meses_con_gap) == 0:
    print("✅ Serie continua — sin gaps mensuales")
else:
    print(f"❌ GAPS DETECTADOS ({len(meses_con_gap)}):")
    print(meses_con_gap)

# 2. Shape esperado
shape_ok = serie_0km.shape == (41, 7)
print(f"{'✅' if shape_ok else '❌'} Shape: {serie_0km.shape} (esperado: (41, 7))")

# 3. Nulos
nulos = serie_0km.isnull().sum().sum()
print(f"{'✅' if nulos == 0 else '❌'} Nulos en la serie: {nulos}")

# 4. Rango de antigüedad — en 0km debería ser ≈ 0 o muy bajo
ant_max = serie_0km['antiguedad_media'].max()
ant_min = serie_0km['antiguedad_media'].min()
print(f"\nAntigüedad media — rango: [{ant_min:.2f}, {ant_max:.2f}] años")
if ant_max > 2:
    print(f"  ⚠️  Antigüedad máxima > 2 años — verificar registros tardíos o errores en anio_modelo")
else:
    print(f"  ✅ Antigüedad consistente con inscripciones iniciales (≈ 0)")

# 5. Volumen total mensual — estadísticas descriptivas
print(f"\nVolumen mensual (total):")
print(serie_0km['total'].describe().round(0).to_string())

# 6. Protocolo 21 en 0km — vehículos nuevos Mercosur
print("pct_protocolo21 en 0km (vehículos nuevos Mercosur):")
print(f"  Rango serie completa: [{serie_0km['pct_protocolo21'].min():.2f}%, {serie_0km['pct_protocolo21'].max():.2f}%]")
print()
print("  Promedio anual:")
print(serie_0km.groupby('anio')['pct_protocolo21'].mean().round(2).to_string())
print()
print("  Referencia usados (NB01):")
referencia = {2023: 40.22, 2024: 39.31, 2025: 41.39, 2026: 42.16}
for anio, val in referencia.items():
    print(f"    {anio}: {val:.2f}%")

print("\n" + "=" * 60)

VALIDACIÓN — SERIE 0KM
✅ Serie continua — sin gaps mensuales
✅ Shape: (41, 7) (esperado: (41, 7))
✅ Nulos en la serie: 0

Antigüedad media — rango: [0.04, 0.14] años
  ✅ Antigüedad consistente con inscripciones iniciales (≈ 0)

Volumen mensual (total):
count       41.0
mean     42247.0
std      11749.0
min      18760.0
25%      34929.0
50%      42094.0
75%      49549.0
max      69752.0
pct_protocolo21 en 0km (vehículos nuevos Mercosur):
  Rango serie completa: [17.29%, 52.17%]

  Promedio anual:
anio
2023    26.24
2024    36.59
2025    46.82
2026    41.58

  Referencia usados (NB01):
    2023: 40.22%
    2024: 39.31%
    2025: 41.39%
    2026: 42.16%



In [11]:
# Promedios anuales de volumen 0km
promedios = serie_0km.groupby('anio')['total'].mean().round(0)
print("Volumen promedio mensual de 0km por año:")
print(promedios.to_string())
print()

# Máximos y mínimos de cada año
print("Mes con mayor volumen por año:")
for anio, grupo in serie_0km.groupby('anio'):
    idx_max = grupo['total'].idxmax()
    print(f"  {anio}: {grupo.loc[idx_max, 'periodo']} — {grupo.loc[idx_max, 'total']:,}")

print()
print("Mes con menor volumen por año:")
for anio, grupo in serie_0km.groupby('anio'):
    idx_min = grupo['total'].idxmin()
    print(f"  {anio}: {grupo.loc[idx_min, 'periodo']} — {grupo.loc[idx_min, 'total']:,}")

Volumen promedio mensual de 0km por año:
anio
2023    37636.0
2024    34678.0
2025    51214.0
2026    49959.0

Mes con mayor volumen por año:
  2023: 2023-01 — 50,601
  2024: 2024-10 — 44,644
  2025: 2025-01 — 69,752
  2026: 2026-01 — 66,737

Mes con menor volumen por año:
  2023: 2023-12 — 18,760
  2024: 2024-12 — 21,976
  2025: 2025-12 — 24,317
  2026: 2026-02 — 42,273


In [12]:
# Evolución de Protocolo 21 en 0km vs usados
print("pct_protocolo21 promedio por año (0km):")
print(serie_0km.groupby('anio')['pct_protocolo21'].mean().round(4).to_string())
print()
print("(Referencia — pct_protocolo21 promedio por año en USADOS [NB01]):")
print("  2023: ~40.2%")
print("  2024: ~39.3%  (baja transitoria durante shock)")
print("  2025: ~41.3%")
print("  2026 (ene-may): ~42.1%")

pct_protocolo21 promedio por año (0km):
anio
2023    26.2367
2024    36.5856
2025    46.8175
2026    41.5784

(Referencia — pct_protocolo21 promedio por año en USADOS [NB01]):
  2023: ~40.2%
  2024: ~39.3%  (baja transitoria durante shock)
  2025: ~41.3%
  2026 (ene-may): ~42.1%


---
## Persistencia

In [10]:
output_path = PATHS["processed"] / "serie_0km.parquet"
serie_0km.to_parquet(output_path, index=False)
print(f"Guardado en: {output_path}")
print(f"Tamaño: {output_path.stat().st_size / 1024:.1f} KB")

Guardado en: c:\Users\usuario\Documents\PORTFOLIO\mercado-limones-causal\data\processed\serie_0km.parquet
Tamaño: 6.6 KB


---
## Hallazgos — Lo que la serie de 0km revela



### 1. Volumen — salto estructural en 2025, no en 2024

Volumen promedio mensual por año:
- 2023: 37.636
- 2024: 34.678  ← caída durante shock Milei
- 2025: 51.214  ← salto (+47% vs 2024)
- 2026: 49.959  (ene-may)

La desregulación de importaciones no se tradujo en volumen durante 2024 —
el mercado de 0km cayó junto con el shock macroeconómico. El salto ocurre
en 2025, cuando la oferta importada se normaliza y el crédito se reactiva.
**Implicancia para NB03:** el lag entre sobrestock de 0km y deterioro de
usados debe buscarse a partir de 2025, no de 2024.

### 2. Estacionalidad de diciembre — patrón opuesto al de usados

Mínimo de volumen en diciembre, todos los años:
- 2023-12: 18.760 (mínimo absoluto de la serie)
- 2024-12: 21.976
- 2025-12: 24.317

En usados, diciembre es el mínimo de antigüedad por presión de objetivos
comerciales (Jensen & Meckling, 1976). En 0km ocurre lo contrario:
diciembre es el mínimo de patentamientos, no el máximo.
La hipótesis de "presión de fin de año genera pico de 0km en diciembre"
queda refutada por los datos. El pico de 0km ocurre en enero:
- 2023-01: 50.601
- 2025-01: 69.752 (máximo absoluto)
- 2026-01: 66.737

Patrón consistente con el calendario de importaciones: los vehículos
que llegan a puerto en el último trimestre se patentan en enero.
**Variable dummy estacional:** en NB03 controlar enero, no diciembre,
para la serie de 0km.

### 3. Protocolo 21 — tendencia ascendente pronunciada y luego corrección

Participación de vehículos Mercosur en inscripciones iniciales:
- 2023: 26.24%
- 2024: 36.59%  (+10 puntos — primera ola de desregulación)
- 2025: 46.82%  (+10 puntos adicionales — consolidación)
- 2026: 41.58%  (corrección — posible saturación o ajuste arancelario)

La tendencia es mucho más pronunciada en 0km que en usados (donde el
rango es 39-42%). Esto confirma que la desregulación entró primero por
el canal nuevo antes de filtrarse al mercado secundario.
**Implicancia para NB03:** pct_protocolo21 en 0km es candidata a
variable de control macroeconómico para separar efecto desregulación
del efecto selección adversa en el modelo ADL.

### 4. Hipótesis causal — lag aparente pendiente de formalización

El salto de volumen en 0km ocurre en 2025. En la serie de usados (NB01)
el ISA muestra su nivel más alto en 2024-01 (antigüedad 15.13 años) y
una tendencia ascendente en pct_protocolo21 desde 2025.
La relación temporal entre ambas series se formalizará en NB03 mediante
CCF y Test de Granger. No se asume causalidad hasta ese análisis.